# Week 5 — Hands-On: Supervised Learning in Depth

Compare regularised linear models, Random Forests, and Gradient Boosting on the same dataset. Build a full scikit-learn pipeline with hyperparameter tuning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

## 1. Load Dataset (California Housing)

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.shape)
df.head()

In [ ]:
X = housing.data
y = housing.target  # Median house value (100k USD)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 2. Baseline — Ridge & Lasso

In [ ]:
results = {}

for name, model in [('Ridge', Ridge(alpha=1.0)), ('Lasso', Lasso(alpha=0.01))]:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2   = r2_score(y_test, y_pred)
    results[name] = {'RMSE': rmse, 'R2': r2}
    print(f'{name:6s}  RMSE={rmse:.4f}  R²={r2:.4f}')

## 3. Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rmse_rf = mean_squared_error(y_test, y_pred_rf, squared=False)
r2_rf   = r2_score(y_test, y_pred_rf)
results['RandomForest'] = {'RMSE': rmse_rf, 'R2': r2_rf}
print(f'RandomForest  RMSE={rmse_rf:.4f}  R²={r2_rf:.4f}')

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', figsize=(7, 4))
plt.title('Random Forest — Feature Importances')
plt.tight_layout()
plt.show()

## 4. Gradient Boosting + RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth'   : [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample'   : [0.7, 0.8, 1.0],
}

gbm = GradientBoostingRegressor(random_state=42)
search = RandomizedSearchCV(
    gbm, param_dist, n_iter=15, cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, random_state=42, verbose=1
)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)

y_pred_gbm = search.predict(X_test)
rmse_gbm = mean_squared_error(y_test, y_pred_gbm, squared=False)
r2_gbm   = r2_score(y_test, y_pred_gbm)
results['GBM (tuned)'] = {'RMSE': rmse_gbm, 'R2': r2_gbm}
print(f'GBM (tuned)  RMSE={rmse_gbm:.4f}  R²={r2_gbm:.4f}')

## 5. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df.round(4))

results_df['RMSE'].sort_values().plot(kind='bar', figsize=(7, 4), color='steelblue')
plt.ylabel('RMSE (lower is better)')
plt.title('Model Comparison on California Housing')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 6. Exercises

1. Add `sklearn.linear_model.ElasticNet` to the comparison. How does it perform?
2. Plot **learning curves** for the best model using `sklearn.model_selection.learning_curve`.
3. Use `GridSearchCV` to tune the `Ridge` `alpha` parameter over `[0.001, 0.01, 0.1, 1, 10, 100]`.
4. *Bonus*: add `XGBoostRegressor` if `xgboost` is installed and compare results.